# Fall Detection - Kaggle Training Pipeline

This notebook contains the complete pipeline for training the Fall Detection model on Kaggle.


## 1. Imports and Installation


In [ ]:
!pip install -q scikit-learn pandas tf-keras


In [ ]:
import json
import sys
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd


## 2. Configuration


In [ ]:
class config:
    """Central configuration for ver2 fall-detection pipeline."""
    
    ROOT = Path("/kaggle/working")
    DATA_SPLITS = Path("/kaggle/input/fall-dataset6/splits")
    
    # Data layout
    DATA_RAW = ROOT / "data" / "raw"
    DATA_SPLITS = ROOT / "data" / "splits"
    MANIFEST_PATH = ROOT / "data" / "video_split_manifest.csv"
    SPLIT_STATS_PATH = ROOT / "data" / "split_stats.json"
    
    # Training artifacts
    ARTIFACTS_DIR = ROOT / "artifacts"
    REPORT_DIR = ROOT / "report"
    
    # Sequence / features (must match deploy/app.py)
    INPUT_TIMESTEPS = 30
    NUM_KEYPOINTS = 17
    NUM_FEATURES = NUM_KEYPOINTS * 3  # x, y, visibility
    
    KEYPOINT_NAMES = [
        "Nose", "Left Eye", "Right Eye", "Left Ear", "Right Ear",
        "Left Shoulder", "Right Shoulder", "Left Elbow", "Right Elbow",
        "Left Wrist", "Right Wrist", "Left Hip", "Right Hip",
        "Left Knee", "Right Knee", "Left Ankle", "Right Ankle",
    ]
    SORTED_KEYPOINT_NAMES = sorted(KEYPOINT_NAMES)
    KEYPOINT_DICT = {name: i for i, name in enumerate(SORTED_KEYPOINT_NAMES)}
    
    MIN_KEYPOINT_CONFIDENCE = 0.3
    
    # Train/val/test ratios (video-level)
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    SPLIT_RANDOM_STATE = 42
    
    # Transformer hyperparameters
    NUM_ENCODER_BLOCKS = 3
    D_MODEL = 64
    NUM_HEADS = 4
    FF_DIM = D_MODEL * 2
    PROJECTION_DIM = D_MODEL
    FINAL_DENSE_UNITS = 32
    DROPOUT_RATE = 0.1
    LEARNING_RATE = 5e-4
    
    # LSTM baseline
    LSTM_UNITS = 64
    LSTM_DROPOUT = 0.2
    
    # Training
    BATCH_SIZE = 32
    EPOCHS = 60
    EARLY_STOPPING_PATIENCE = 15
    
    # Inference / deployment
    DEFAULT_THRESHOLD = 0.5
    TFLITE_MODEL_NAME = "fall_detection_transformer.tflite"
    SAVED_MODEL_DIR = "fall_model_exported_sm"
    


## 3. Data Loader


In [ ]:
"""Load .npy skeleton sequences from split folders."""
from __future__ import annotations

import os
from glob import glob
from pathlib import Path

import numpy as np



def expected_shape() -> tuple[int, int]:
    return config.INPUT_TIMESTEPS, config.NUM_FEATURES


def load_dataset(
    data_path: str | Path,
    normalize: bool = True,
    min_confidence: float = config.MIN_KEYPOINT_CONFIDENCE,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    data_path = Path(data_path)
    exp_shape = expected_shape()
    x_list: list[np.ndarray] = []
    y_list: list[int] = []
    paths: list[str] = []

    print(f"Loading from {data_path}, expected shape {exp_shape}")
    for label_name, label_val in [("no_fall", 0), ("fall", 1)]:
        folder = data_path / label_name
        if not folder.is_dir():
            print(f"  Warning: missing folder {folder}")
            continue
        files = sorted(glob(str(folder / "*.npy")))
        loaded = 0
        for fp in files:
            try:
                arr = np.load(fp)
            except Exception as e:
                print(f"  Warning: cannot load {fp}: {e}")
                continue
            if arr.shape != exp_shape:
                print(f"  Warning: skip {fp} shape {arr.shape} != {exp_shape}")
                continue
            if normalize:
                arr = normalize_skeleton(arr, min_confidence=min_confidence)
                if np.isnan(arr).any():
                    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            x_list.append(arr.astype(np.float32))
            y_list.append(label_val)
            paths.append(fp)
            loaded += 1
        print(f"  {label_name}: {loaded}/{len(files)} sequences")

    if not x_list:
        return np.array([]), np.array([]), []
    return np.stack(x_list), np.array(y_list, dtype=np.float32), paths



## 4. Models (LSTM & Transformer)


In [ ]:
"""LSTM baseline for skeleton sequences."""
from __future__ import annotations

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam



def create_lstm_classifier(
    input_shape: tuple[int, int] | None = None,
    lstm_units: int = config.LSTM_UNITS,
    dropout_rate: float = config.LSTM_DROPOUT,
    learning_rate: float = config.LEARNING_RATE,
) -> tf.keras.Model:
    if input_shape is None:
        input_shape = (config.INPUT_TIMESTEPS, config.NUM_FEATURES)

    f1_macro = tf.keras.metrics.F1Score(
        average="macro",
        threshold=0.5,
        name="f1_macro",
    )
    model = Sequential(name="lstm_fall_classifier")
    model.add(LSTM(lstm_units, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(dropout_rate))
    model.add(LSTM(lstm_units // 2))
    model.add(Dropout(dropout_rate))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", f1_macro],
    )
    return model



In [ ]:
"""Transformer classifier for skeleton sequences."""
from __future__ import annotations

import tensorflow as tf
from tensorflow.keras.layers import (
    Add,
    Dense,
    Dropout,
    Embedding,
    GlobalAveragePooling1D,
    Input,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam



def transformer_encoder_block(
    inputs,
    d_model: int,
    num_heads: int,
    ff_dim: int,
    dropout_rate: float = 0.1,
    name_prefix: str = "",
):
    attn = MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout_rate,
        name=f"{name_prefix}_mha",
    )(inputs, inputs, inputs)
    attn = Dropout(dropout_rate, name=f"{name_prefix}_mha_dropout")(attn)
    out1 = LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_layernorm1")(inputs + attn)

    ffn = Dense(ff_dim, activation="relu", name=f"{name_prefix}_ffn_dense1")(out1)
    ffn = Dense(d_model, name=f"{name_prefix}_ffn_dense2")(ffn)
    ffn = Dropout(dropout_rate, name=f"{name_prefix}_ffn_dropout")(ffn)
    out2 = LayerNormalization(epsilon=1e-6, name=f"{name_prefix}_layernorm2")(out1 + ffn)
    return out2


def positional_embedding(seq_len: int, d_model: int, name_prefix: str = ""):
    positions = tf.range(start=0, limit=seq_len, delta=1)
    pos_2d = Embedding(
        input_dim=seq_len,
        output_dim=d_model,
        name=f"{name_prefix}_pos_embed",
    )(positions)
    return tf.expand_dims(pos_2d, axis=0)


def create_transformer_classifier(
    input_shape: tuple[int, int] | None = None,
    num_encoder_blocks: int = config.NUM_ENCODER_BLOCKS,
    d_model: int = config.D_MODEL,
    num_heads: int = config.NUM_HEADS,
    ff_dim: int = config.FF_DIM,
    projection_dim: int | None = None,
    final_dense_units: int = config.FINAL_DENSE_UNITS,
    dropout_rate: float = config.DROPOUT_RATE,
    learning_rate: float = config.LEARNING_RATE,
) -> Model:
    if input_shape is None:
        input_shape = (config.INPUT_TIMESTEPS, config.NUM_FEATURES)
    if projection_dim is None:
        projection_dim = d_model

    timesteps, _ = input_shape
    inputs = Input(shape=input_shape, name="input_features")
    x = Dense(projection_dim, name="feature_projection")(inputs)
    pos = positional_embedding(timesteps, projection_dim, name_prefix="pos_enc")
    x = Add(name="add_positional_encoding")([x, pos])
    x = Dropout(dropout_rate, name="input_dropout_after_pos_enc")(x)

    for i in range(num_encoder_blocks):
        x = transformer_encoder_block(
            x,
            projection_dim,
            num_heads,
            ff_dim,
            dropout_rate,
            name_prefix=f"encoder_block_{i + 1}",
        )

    x = GlobalAveragePooling1D(name="global_avg_pooling")(x)
    x = Dropout(dropout_rate, name="dropout_after_pooling")(x)
    x = Dense(final_dense_units, activation="relu", name="final_dense_1")(x)
    x = Dropout(dropout_rate / 2, name="dropout_final_dense")(x)
    outputs = Dense(1, activation="sigmoid", name="output_sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs)
    f1_macro = tf.keras.metrics.F1Score(
        average="macro",
        threshold=0.5,
        name="f1_macro",
    )
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", f1_macro],
    )
    return model



## 5. Metrics & Export TFLite


In [ ]:
"""Evaluation metrics and threshold tuning."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    precision_recall_fscore_support,
)


@dataclass
class EvalResult:
    threshold: float
    accuracy: float
    precision_fall: float
    recall_fall: float
    f1_fall: float
    f2_fall: float
    confusion: np.ndarray
    report: str


def predict_labels(probs: np.ndarray, threshold: float) -> np.ndarray:
    return (probs.reshape(-1) >= threshold).astype(int)


def evaluate_at_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    threshold: float,
) -> EvalResult:
    y_pred = predict_labels(probs, threshold)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average="binary", zero_division=0
    )
    f2 = fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0)
    report = classification_report(
        y_true, y_pred, target_names=["no_fall", "fall"], zero_division=0
    )
    return EvalResult(
        threshold=threshold,
        accuracy=float(accuracy_score(y_true, y_pred)),
        precision_fall=float(prec),
        recall_fall=float(rec),
        f1_fall=float(f1),
        f2_fall=float(f2),
        confusion=cm,
        report=report,
    )


def find_best_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    metric: str = "f2_fall",
    thresholds: np.ndarray | None = None,
) -> EvalResult:
    if thresholds is None:
        thresholds = np.arange(0.1, 0.91, 0.01)
    best: EvalResult | None = None
    for t in thresholds:
        result = evaluate_at_threshold(y_true, probs, float(t))
        score = getattr(result, metric)
        if best is None or score > getattr(best, metric):
            best = result
    assert best is not None
    return best


def error_analysis_df(
    y_true: np.ndarray,
    probs: np.ndarray,
    filepaths: list[str],
    threshold: float,
    split_name: str,
) -> pd.DataFrame:
    y_pred = predict_labels(probs, threshold)
    rows = []
    for i, fp in enumerate(filepaths):
        true_l = "fall" if y_true[i] == 1 else "no_fall"
        pred_l = "fall" if y_pred[i] == 1 else "no_fall"
        err_type = "TP" if y_true[i] == 1 and y_pred[i] == 1 else ""
        if y_true[i] == 0 and y_pred[i] == 1:
            err_type = "FP"
        elif y_true[i] == 1 and y_pred[i] == 0:
            err_type = "FN"
        elif y_true[i] == 0 and y_pred[i] == 0:
            err_type = "TN"
        rows.append(
            {
                "File Name": Path(fp).name,
                "True": true_l,
                "Pred": pred_l,
                "Prob": float(probs.reshape(-1)[i]),
                "Type": err_type,
                "Set": split_name,
            }
        )
    return pd.DataFrame(rows)


def metrics_summary_row(model_name: str, split_name: str, result: EvalResult) -> dict:
    return {
        "Model": model_name,
        "Split": split_name,
        "Threshold": result.threshold,
        "Accuracy": result.accuracy,
        "Precision_fall": result.precision_fall,
        "Recall_fall": result.recall_fall,
        "F1_fall": result.f1_fall,
        "F2_fall": result.f2_fall,
    }



In [ ]:
"""Export Keras model to TFLite."""
from __future__ import annotations

from pathlib import Path

import tensorflow as tf



def export_to_tflite(
    model: tf.keras.Model,
    export_dir: Path | None = None,
    tflite_path: Path | None = None,
) -> Path:
    export_dir = export_dir or (config.ARTIFACTS_DIR / config.SAVED_MODEL_DIR)
    tflite_path = tflite_path or (config.ARTIFACTS_DIR / config.TFLITE_MODEL_NAME)

    export_dir = Path(export_dir)
    tflite_path = Path(tflite_path)
    export_dir.mkdir(parents=True, exist_ok=True)
    tflite_path.parent.mkdir(parents=True, exist_ok=True)

    model.export(str(export_dir))
    converter = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
    tflite_bytes = converter.convert()
    tflite_path.write_bytes(tflite_bytes)
    size_kb = len(tflite_bytes) / 1024
    print(f"Saved TFLite: {tflite_path} ({size_kb:.2f} KB)")
    return tflite_path


def verify_tflite(tflite_path: Path) -> dict:
    interp = tf.lite.Interpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    info = {
        "input_shape": inp["shape"],
        "input_dtype": str(inp["dtype"]),
        "output_shape": out["shape"],
        "output_dtype": str(out["dtype"]),
    }
    expected = [1, config.INPUT_TIMESTEPS, config.NUM_FEATURES]
    if list(inp["shape"]) != expected:
        raise ValueError(f"TFLite input {inp['shape']} != expected {expected}")
    return info



## 6. Training Functions


In [ ]:
def train_model(
    model: tf.keras.Model,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    model_tag: str,
    epochs: int,
    batch_size: int,
) -> tf.keras.Model:
    callbacks = [
        EarlyStopping(
            monitor="val_f1_macro",
            patience=config.EARLY_STOPPING_PATIENCE,
            mode="max",
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_f1_macro",
            factor=0.5,
            patience=5,
            mode="max",
            min_lr=1e-6,
            verbose=1,
        ),
    ]
    y_train_2d = y_train.reshape(-1, 1)
    y_val_2d = y_val.reshape(-1, 1)
    history = model.fit(
        x_train,
        y_train_2d,
        validation_data=(x_val, y_val_2d),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )
    hist_path = config.ARTIFACTS_DIR / f"{model_tag}_history.json"
    hist_path.parent.mkdir(parents=True, exist_ok=True)
    serializable = {k: [float(v) for v in vals] for k, vals in history.history.items()}
    hist_path.write_text(json.dumps(serializable, indent=2), encoding="utf-8")
    return model


## 7. Execution (Train & Evaluate)


In [ ]:

# KAGGLE TRAINING EXECUTION
print("Loading datasets from Kaggle input path...")
# Change this path to match your Kaggle dataset path after uploading
KAGGLE_DATA_DIR = Path("/kaggle/input/fall-dataset6/splits")

# If dataset is uploaded directly:
if not KAGGLE_DATA_DIR.exists():
    print(f"Warning: {KAGGLE_DATA_DIR} not found. Please update KAGGLE_DATA_DIR to point to your dataset splits.")
else:
    config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    config.REPORT_DIR.mkdir(parents=True, exist_ok=True)

    x_train, y_train, _ = load_dataset(KAGGLE_DATA_DIR / "train")
    x_val, y_val, _ = load_dataset(KAGGLE_DATA_DIR / "val")
    x_test, y_test, test_paths = load_dataset(KAGGLE_DATA_DIR / "test")

    print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

    # Choose model: 'transformer' or 'lstm'
    MODEL_TYPE = 'transformer'

    if MODEL_TYPE == "lstm":
        model = create_lstm_classifier()
        model_tag = "lstm"
        keras_path = config.ARTIFACTS_DIR / "lstm_best.keras"
    else:
        model = create_transformer_classifier()
        model_tag = "transformer"
        keras_path = config.ARTIFACTS_DIR / "transformer_best.keras"

    model = train_model(
        model, x_train, y_train, x_val, y_val, model_tag, config.EPOCHS, config.BATCH_SIZE
    )
    model.save(keras_path)
    print(f"Saved Keras model: {keras_path}")

    val_probs = model.predict(x_val, verbose=0).reshape(-1)
    best_val = find_best_threshold(y_val, val_probs, metric="f2_fall")
    print(f"\nBest threshold on VAL: {best_val.threshold:.2f}")
    print(best_val.report)

    threshold_path = config.ARTIFACTS_DIR / f"{model_tag}_threshold.json"
    threshold_path.write_text(
        json.dumps({"threshold": best_val.threshold, "metric": "f2_fall"}, indent=2),
        encoding="utf-8",
    )

    if len(x_test) > 0:
        test_probs = model.predict(x_test, verbose=0).reshape(-1)
        test_res = evaluate_at_threshold(y_test, test_probs, best_val.threshold)
        print(f"\nTEST @ threshold {best_val.threshold:.2f}:")
        print(test_res.report)

    if MODEL_TYPE == "transformer":
        tflite_path = export_to_tflite(model)
        verify_tflite(tflite_path)
        print(f"TFLite exported successfully to: {tflite_path}")

